# 05 — Stain normalization

RocqiPath supports Reinhard, Macenko, and Vahadane normalization through a
common train/apply workflow. Training writes reusable `.npz` weights;
application preserves each input filename under the shallow
`stain_normalization` output module.

Install `.[stain]`. Choose representative training patches from the same
stain/channel and avoid blank background, folds, pen, and severe artifacts.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def find_project_root(start: Path | None = None) -> Path:
    '''Find the RocqiPath repository whether Jupyter starts at root or how_to_use.'''
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "rocqipath"
        ).is_dir():
            return candidate
    raise FileNotFoundError(
        "RocqiPath repository not found. Start Jupyter inside the cloned repository."
    )


PROJECT_ROOT = find_project_root()
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

DATA_ROOT = PROJECT_ROOT / "data"
RESULTS_ROOT = PROJECT_ROOT / "results"

print(f"Project : {PROJECT_ROOT}")
print(f"Data    : {DATA_ROOT}")
print(f"Results : {RESULTS_ROOT}")


In [ ]:
PATCH_ROOT = RESULTS_ROOT / "patch_extraction"
NORMALIZATION_ROOT = RESULTS_ROOT

NORMALIZER = "macenko"  # "reinhard", "macenko", or "vahadane"

# discover_files filters when a requested token is an exact path component.
# Use ["all"] for a pre-filtered directory, or a folder token such as ["he"].
STAIN_TOKENS = ["all"]

RUN_TRAIN = False
RUN_APPLY = False


## Algorithm choice

| Method | Strength | Practical note |
|---|---|---|
| Reinhard | Fast color-statistics matching | Useful baseline; train incrementally |
| Macenko | Optical-density stain separation | Common H&E choice; fit a representative mosaic |
| Vahadane | Sparse stain separation | Often preserves structure well; heaviest dependency path |

Use one trained weight file for a coherent cohort. Do not fit a separate
normalizer per patch or per outcome group.


In [ ]:
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
patch_files = (
    sorted(
        path for path in PATCH_ROOT.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )
    if PATCH_ROOT.is_dir()
    else []
)

print(f"Candidate images: {len(patch_files)}")
for path in patch_files[:10]:
    print(" ", path.relative_to(PATCH_ROOT))


In [ ]:
from rocqipath.config import StainNormalizationConfig

stain_cfg = StainNormalizationConfig(
    n_type=NORMALIZER,
    stains=STAIN_TOKENS,
    fit_min_tissue=0.10,
    max_train_patches=500,
    resume=True,
    weights_path=None,
)

print(stain_cfg.to_dict())


## Train weights

Macenko/Vahadane resize accepted training patches to 256×256, build a
deterministic mosaic, and cap the sample using
`max_train_patches`. Reinhard accumulates color statistics from accepted
patches.


In [ ]:
from rocqipath.stain import (
    run_stain_normalization_apply,
    run_stain_normalization_train,
)

if RUN_TRAIN:
    if not PATCH_ROOT.is_dir():
        raise FileNotFoundError(PATCH_ROOT)
    weights_path = run_stain_normalization_train(
        input_dir=str(PATCH_ROOT),
        output_dir=str(NORMALIZATION_ROOT),
        cfg=stain_cfg,
    )
    print(f"Weights: {weights_path}")
else:
    weights_path = (
        NORMALIZATION_ROOT
        / "stain_normalization"
        / f"{NORMALIZER}_weights.npz"
    )
    print("Set RUN_TRAIN=True to fit weights.")
    print(f"Expected weights: {weights_path}")


## Apply weights

Set `weights_path` explicitly when applying weights trained elsewhere.
With `resume=True`, existing normalized files are skipped.


In [ ]:
apply_cfg = StainNormalizationConfig.from_dict(
    {
        **stain_cfg.to_dict(),
        "weights_path": str(weights_path),
        "resume": True,
    }
)

if RUN_APPLY:
    normalization_summary = run_stain_normalization_apply(
        input_dir=str(PATCH_ROOT),
        output_dir=str(NORMALIZATION_ROOT),
        cfg=apply_cfg,
    )
    print(normalization_summary)
else:
    normalization_summary = None
    print("Set RUN_APPLY=True after a weight file exists.")


## Visual before/after QC

Inspect multiple slides and tissue phenotypes. Normalization should reduce
stain variability without erasing nuclei, DAB signal, gland boundaries, or
texture. A visually uniform image can still be scientifically distorted.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

normalized_root = NORMALIZATION_ROOT / "stain_normalization"
normalized_files = (
    sorted(
        path for path in normalized_root.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )
    if normalized_root.is_dir()
    else []
)

if patch_files and normalized_files:
    original_path = patch_files[0]
    same_name = [p for p in normalized_files if p.name == original_path.name]
    normalized_path = same_name[0] if same_name else normalized_files[0]

    original = np.array(Image.open(original_path).convert("RGB"))
    normalized = np.array(Image.open(normalized_path).convert("RGB"))

    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(original)
    axes[0].set_title(f"Original\n{original_path.name}")
    axes[1].imshow(normalized)
    axes[1].set_title(f"Normalized ({NORMALIZER})\n{normalized_path.name}")
    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    print("Original RGB mean  :", original.mean(axis=(0, 1)).round(1))
    print("Normalized RGB mean:", normalized.mean(axis=(0, 1)).round(1))
else:
    print("Run training/application before visual QC.")


## Frequent issues

- **No patches found**: `stains` matches exact path components; use
  `["all"]` when the input directory is already filtered.
- **No tissue passed**: lower `fit_min_tissue` only after inspecting the
  input patches for excessive background.
- **Weights missing**: train first or set an explicit `weights_path`.
- **Color cast or lost morphology**: improve the training cohort and compare
  algorithms; do not merely tune until one example looks attractive.
- **Data leakage**: fit normalization on the training split and reuse those
  weights for validation/test data.
